[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafamayo/Workshop-UPNA-2026/blob/main/labs/day2/ai-mini-module/ai-notebook-extended.ipynb)

# Extended AI Mini-Module — Simulation, Visualization, ML-ready feature extraction
UPNA Workshop 2026 — Day 2

This notebook shows basic AI-like activities:

- Getting data using FHIR
- Visualization of vital-sign trends
- Preparing data for ML algorithms
- Error-handling exercises (optional)

The goal is to deepen understanding of **AI preprocessing**, **data quality**, and **workflow design**, not to build a real clinical model.

## 1. Setup

In [ ]:
!pip install requests

# !pip install pandas
# when working on jupyter.org use this instead
!mamba install pandas

# !pip install scikit-learn
# when working on jupyter.org use this instead
!mamba install scikit-learn

In [ ]:
import requests, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

FHIR_SERVER = "https://hapi.fhir.org/baseR4/"

## 2. Connect to a Patient

Post a bundle to create a patient with several observations.

In [ ]:
# use the bundle in the file bundle.json
bundle = {...}


In [ ]:
resp = requests.post(
    FHIR_SERVER,  # transaction endpoint is often the base URL
    headers={"Content-Type": "application/fhir+json", "Accept": "application/fhir+json"},
    data=json.dumps(bundle)
)

print(resp.status_code)
result_bundle = resp.json()
result_bundle

In [ ]:
patient_id = "131276553"  # Insert manually if using real FHIR data
patient_id

### Helper functions

In [ ]:
def get_observations(patient_id, loinc_code):
    """
    Fetch Observations for the patient filtered by LOINC code.
    Returns a list of Observation resources.
    """
    url = f"{FHIR_SERVER}Observation?subject=Patient/{patient_id}&code={loinc_code}"
    r = requests.get(url, headers={"Accept": "application/fhir+json"})
    data = r.json()
    if "entry" not in data:
        return []
    return [e["resource"] for e in data["entry"]]


def extract_value(obs):
    """Try to extract a single numeric valueQuantity from an Observation."""
    try:
        return obs["valueQuantity"]["value"]
    except KeyError:
        return np.nan


def obs_to_dataframe(observations):
    """Convert a list of Observations to a tidy DataFrame."""
    rows = []
    for obs in observations:
        value = extract_value(obs)
        time = obs.get("effectiveDateTime", None)
        rows.append({"value": value, "time": time})
    return pd.DataFrame(rows)

## 3. Retrieve Vitals
We use common LOINC codes:
- Heart rate: `8867-4`
- Temperature: `8310-5`
- Oxygen saturation in Arterial blood by Pulse oximetry: `59408-5`


In [ ]:
# Replace patient_id above before running this

heart_rate_obs = get_observations(patient_id, "8867-4")
temperature_obs = get_observations(patient_id, "8310-5")
oxygen_saturation_obs = get_observations(patient_id, "59408-5")

len(heart_rate_obs), len(temperature_obs), len(oxygen_saturation_obs)

### Convert to DataFrames

In [ ]:
df_hr = obs_to_dataframe(heart_rate_obs)
df_temp = obs_to_dataframe(temperature_obs)
df_oxy = obs_to_dataframe(oxygen_saturation_obs)

df_hr.head(), df_temp.head(), df_oxy.head() 

### Put them together in a single DataFrame including a time index

In [ ]:
# Make sure they are sorted consistently (just in case)
df_hr = df_hr.sort_values("time").reset_index(drop=True)
df_temp = df_temp.sort_values("time").reset_index(drop=True)
df_oxy = df_oxy.sort_values("time").reset_index(drop=True)

# Create synthetic time index
n = min(len(df_hr), len(df_temp), len(df_oxy))

df_vitals = pd.DataFrame({
    "time": range(n),
    "heart_rate": df_hr["value"].iloc[:n].values,
    "temperature": df_temp["value"].iloc[:n].values,
    "oxygen_saturation": df_oxy["value"].iloc[:n].values
})

df_vitals.head()

## 4. Visualization of Vital-Sign Trends

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Heart Rate
axes[0].plot(df_vitals["time"], df_vitals["heart_rate"])
axes[0].set_ylabel("Heart Rate (bpm)")
axes[0].set_title("Vital Signs Over Time")

# Temperature
axes[1].plot(df_vitals["time"], df_vitals["temperature"])
axes[1].set_ylabel("Temperature (°C)")

# Oxygen Saturation
axes[2].plot(df_vitals["time"], df_vitals["oxygen_saturation"])
axes[2].set_ylabel("Oxygen Saturation (%)")
axes[2].set_xlabel("Time Index")

plt.tight_layout()
plt.show()

## 5. Extract features and combine them into a Single Feature Vector
We compute simple features like mean, last value, and variability.

In [ ]:
features = {
    "hr_mean": df_vitals.heart_rate.mean(),
    "hr_std": df_vitals.heart_rate.std(),
    "hr_last": df_vitals.heart_rate.iloc[-1],
    "temp_mean": df_vitals.temperature.mean(),
    "temp_std": df_vitals.temperature.std(),
    "temp_last": df_vitals.temperature.iloc[-1],
    "oxy_mean": df_vitals.oxygen_saturation.mean(),
    "oxy_std": df_vitals.oxygen_saturation.std(),
    "oxy_last": df_vitals.oxygen_saturation.iloc[-1]
}
features

## 6. Optional tasks: Error handling

In [ ]:
# Exercise 1 — Handle missing Patient ID
# TODO: Wrap this in try/except and print a clear error message

def load_patient_safe(patient_id):
    try:
        if not patient_id:
            raise ValueError("Patient ID is empty!")
        url = f"{FHIR_SERVER}Patient/{patient_id}"
        r = requests.get(url)
        return r.json()
    except Exception as e:
        print("Error:", e)

load_patient_safe("")

In [ ]:
# Exercise 2 — Invalid Observation query
# TODO: Add error messages for non-200 HTTP responses

def get_obs_safe(patient_id, code):
    try:
        url = f"{FHIR_SERVER}Observation?subject=Patient/{patient_id}&code={code}"
        r = requests.get(url)
        if r.status_code != 200:
            print("Server returned", r.status_code)
            return None
        data = r.json()
        if "entry" not in data:
            print("No Observations found.")
            return []
        return [e["resource"] for e in data["entry"]]
    except Exception as e:
        print("Error during request:", e)

get_obs_safe("12345", "INVALID-CODE")